In [ ]:
# %% 完整高效中文诗歌 Transformer 训练脚本（整合修正版）
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# 1. 数据准备与字符级编码
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f"数据总长度 (字符数): {len(text):,}")

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"词汇表大小: {vocab_size}")
print(f"前50个字符示例: {''.join(chars[:50])}")

# 创建字符到索引的映射
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]  # 编码：字符串 -> 索引列表
decode = lambda l: ''.join([itos[i] for i in l])  # 解码：索引列表 -> 字符串

# 设置设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 将整个文本编码为张量
data = torch.tensor(encode(text), dtype=torch.long, device=device)
n = len(data)
split_idx = int(n * 0.9)
train_data = data[:split_idx]
val_data = data[split_idx:]
print(f"训练集 tokens: {len(train_data):,}")
print(f"验证集 tokens: {len(val_data):,}")

# 2. 超参数配置
batch_size = 64
block_size = 256
max_iters = 200000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200
n_embd = 256
n_head = 4
n_layer = 6
dropout = 0.4
weight_decay = 0.1
torch.manual_seed(1337)

# 3. 数据批处理函数
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x, y

# 4. Transformer 模型定义
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# 5. 初始化模型、优化器
model = BigramLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,weight_decay = 0.1)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# 6. 训练循环
print("开始训练...")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"迭代次数 {iter:4d} | 训练损失 {losses['train']:.4f} | 验证损失 {losses['val']:.4f}")

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# 7. 生成诗词（关键修复：从<START>开始生成）
print("\n" + "="*50)
print("生成诗歌示例：")

# 找到起始标记'<START>'的索引
# 重要：确保上下文以数据集的起始标记开始，这样模型才知道在“开始一首诗”
start_token = encode('<START>')[0]  # 获取'<START>'对应的索引
context = torch.tensor([[start_token]], dtype=torch.long, device=device)  # 形状 (1, 1)


数据总长度 (字符数): 2,200,645
词汇表大小: 5902
前50个字符示例: 
 :<>ADENRST“”、。《》㧑㬠㶉䅉䆉䌷䍠䕷䗖䙀䣔䩮䯄䯳䰀䰄䰉䰐䰒䴔䴖一丁七万丈三上下不与丐
Using device: cuda
训练集 tokens: 1,980,580
验证集 tokens: 220,065
开始训练...
迭代次数    0 | 训练损失 8.7992 | 验证损失 8.7984
迭代次数  500 | 训练损失 4.2646 | 验证损失 4.2185
迭代次数 1000 | 训练损失 3.9472 | 验证损失 3.9351
迭代次数 1500 | 训练损失 3.7664 | 验证损失 3.7918
迭代次数 2000 | 训练损失 3.6227 | 验证损失 3.6595
迭代次数 2500 | 训练损失 3.4764 | 验证损失 3.5462
迭代次数 3000 | 训练损失 3.3842 | 验证损失 3.4808
迭代次数 3500 | 训练损失 3.3139 | 验证损失 3.4206
迭代次数 4000 | 训练损失 3.2504 | 验证损失 3.3895
迭代次数 4500 | 训练损失 3.1935 | 验证损失 3.3589
迭代次数 5000 | 训练损失 3.1484 | 验证损失 3.3347
迭代次数 5500 | 训练损失 3.1087 | 验证损失 3.3013
迭代次数 6000 | 训练损失 3.0704 | 验证损失 3.2880
迭代次数 6500 | 训练损失 3.0378 | 验证损失 3.2753
迭代次数 7000 | 训练损失 3.0047 | 验证损失 3.2549
迭代次数 7500 | 训练损失 2.9749 | 验证损失 3.2504
迭代次数 7999 | 训练损失 2.9466 | 验证损失 3.2228

生成诗歌示例：
<END>

<START>
词牌: 浣溪沙
饮罢征衫带玉杯。
袂娥粘得宽红袖，松江冷泼落梅花。
别后不知春色到，共危机下玉楼空。
<END>

<START>
词牌: 浣溪沙
牵马妆貂衮绣衣。
双前白鹭未随群。
好花须是尽心肠。
轻颦生怕醉，怕人烟雨海棠时。
<END>

<START>
词牌: 浣溪沙
捧劝金卮午睡重。
楼高乍立两三弦。
震机茅楮面，丛花

In [20]:
# 6. 训练循环
print("开始训练...")
for iter in range(2000):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"迭代次数 {iter:4d} | 训练损失 {losses['train']:.4f} | 验证损失 {losses['val']:.4f}")

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


开始训练...
迭代次数    0 | 训练损失 2.6431 | 验证损失 3.1510
迭代次数  500 | 训练损失 2.6229 | 验证损失 3.1590
迭代次数 1000 | 训练损失 2.6064 | 验证损失 3.1456
迭代次数 1500 | 训练损失 2.5910 | 验证损失 3.1520


In [21]:
def fixed_generate(model, start_token, max_tokens=300, temperature=0.8, top_k=50, top_p=0.95, stop_on_end=True):
    """
    修复的生成函数，正确检查<END>标记
    """
    model.eval()
    with torch.no_grad():
        # 获取<START>和<END>的完整token序列
        start_tokens = encode(start_token)
        end_tokens = encode('<END>')
        
        # 初始化上下文
        context = torch.tensor([start_tokens], dtype=torch.long, device=device)
        
        for _ in range(max_tokens):
            # 截断到block_size
            idx_cond = context if context.size(1) <= block_size else context[:, -block_size:]
            
            # 前向传播
            logits, _ = model(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            # Top-k过滤
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            # Top-p（核）采样
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits[indices_to_remove] = -float('Inf')
            
            # 采样
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            
            # 添加到上下文
            context = torch.cat((context, idx_next), dim=1)
            
            # 检查是否生成了完整的<END>标记
            if stop_on_end and context.size(1) >= len(end_tokens):
                # 检查最后len(end_tokens)个token是否等于end_tokens
                recent_tokens = context[0, -len(end_tokens):].tolist()
                if recent_tokens == end_tokens:
                    break
    
    return decode(context[0].tolist())

# 生成示例
print("模型采样：")
for _ in range(5):
    poem = fixed_generate(model, '<START>', max_tokens=200, temperature=0.8, top_k=50, top_p=0.95)
    print(poem)
    print("-" * 50)

模型采样：
<START>
词牌: 小重山
水暖云轻碧玉盘。
寒衣衬，水沈香。
夜深相向宿，无计奈心长。
月明花外影横斜。
闲倚小楼西。
画楼深处倚，帘卷玉钩斜。
<END>
--------------------------------------------------
<START>
词牌: 好事近
帘外卷帘风，暗雨暗香残馥。
月娥花下再相逢，梦里梦中寻觅。
小院小楼空，玉箫声在碧。
玉阶深处觅。
<END>
--------------------------------------------------
<START>
词牌: 临江仙
不数重阳天上草，一番过了轻阴。
天涯南北使君来。
相思无处问，天意与谁同。
一尊今夜问，今夜鹊南枝。
欲知何事也，只有几时休。
<END>
--------------------------------------------------
<START>
词牌: 好事近
花里见梅花，雪未应如客。
不是有情多，为你恼人愁绝。
今年心事更堪怜，把千金、赚却是他宿。
<END>
--------------------------------------------------
<START>
词牌: 玉楼春
春风十里桃花树。
小雨烘晴春欲暮。
绿杨枝上红如雾。
舞燕一双春恨绪。
一饷芳时春又暮。
楼前一饷春闲语。
<END>
--------------------------------------------------
